# Tuned-lens read/write dissociation (faithfulness check)

Replaces the raw logit-lens read with a **tuned-lens** read, keeping everything else fixed (same failed
direct-generation items, same gold first-token targets, same same-relation decoy pools, same hard-decoy
criterion, same final write/rank). This answers the logit-lens-faithfulness worry.

**Tuned lens** here is the translator form: per layer, fit an affine map $A_l h_l + b_l \approx h_{final}$
on a HELD-OUT fit split (so the read is not circular), then read through the TRUE final norm and
unembedding. Probes are fit on `n_fit` items disjoint from the `n_eval` items used to measure the
dissociation.

**Models** (as recommended): Llama-3.1-8B (central, strong effect) and Qwen2.5-3B (the interesting
heterogeneity). Running both lets you report the representative model and check the odd regime.

**Reports:** tuned hard-readable failure rate, fraction of those not top-ranked, tuned read vs final
rank correlation, and the overlap between logit-lens-readable and tuned-lens-readable sets.

**The key result wanted:** even under a tuned lens, a nontrivial subset of failed generations has
fact-specific answer decodability yet the answer is not selected at final readout. If it weakens, the
honest statement is "a smaller, more conservative readable subset, but the dissociation persists within
it."

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`, `tunedlens_core.py`. Run `test_tunedlens_core.py`.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr, tunedlens_core as tl
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=800          # need enough for a fit split + a disjoint eval split
N_FIT=300; N_EVAL=400; L2=1.0

MODELS=["meta-llama/Llama-3.1-8B","Qwen/Qwen2.5-3B"]

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items:",len(ITEMS),"| fit split:",N_FIT,"| eval split:",N_EVAL)

## 1. Fit tuned-lens probes (held-out) and run the dissociation

In [ ]:
TUNED={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        TUNED[name]=mr.exp_tuned_lens_readwrite(ctx, ctx["freq"], n_fit=N_FIT, n_eval=N_EVAL, l2=L2)
        r=TUNED[name]
        print(f"  n_eval_failures={r.get('n')} recon_R2={r.get('mean_recon_r2',float('nan')):.3f}",flush=True)
        print(f"  tuned hard-readable={r.get('tuned_hard_readable_rate',float('nan')):.3f} "
              f"(logit {r.get('logit_hard_readable_rate',float('nan')):.3f}) | "
              f"tuned nottop|readable={r.get('tuned_nottop_among_readable',float('nan')):.3f}",flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); TUNED[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(TUNED,open("tuned_lens_readwrite.json","w"),indent=2,default=float)

## 2. Tuned vs logit lens — the dissociation under both reads
`recon_R2` is the tuned-lens translator quality (should be high, ~0.8+; if low, probes underfit and the
read is weak). The dissociation persists if `tuned_nottop_among_readable` stays high (the answer is
readable under the tuned lens yet still not selected).

In [ ]:
rows=[]
for nm,v in TUNED.items():
    if "status" in v: print(nm,"->",v.get("error")); continue
    rows.append({"model":nm.split("/")[-1],"n_fail":v["n"],"recon_R2":round(v["mean_recon_r2"],3),
                 "tuned_hard_readable":round(v["tuned_hard_readable_rate"],3),
                 "logit_hard_readable":round(v["logit_hard_readable_rate"],3),
                 "tuned_nottop|read":round(v["tuned_nottop_among_readable"],3),
                 "logit_nottop|read":round(v["logit_nottop_among_readable"],3),
                 "rho_tuned":round(v["rho_tunedread_negwrite"],3),
                 "rho_logit":round(v["rho_logitread_negwrite"],3),
                 "readable_overlap":round(v["readable_set_overlap"],3)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nKEY: tuned_hard_readable > 0 with tuned_nottop|read high = even under the tuned lens, a")
print("nontrivial subset of failures is fact-specifically readable yet not selected at final readout.")

## 3. Interpretation guide
- If `tuned_hard_readable` is close to `logit_hard_readable` and `tuned_nottop|read` stays high: the
  dissociation is **not** a logit-lens artifact. Strongest outcome.
- If `tuned_hard_readable` is smaller but still positive with high not-top: report "a smaller, more
  conservative readable subset, but the dissociation persists within it." Still fine.
- If `tuned_hard_readable` ~ 0: the tuned lens removes the readability; you'd narrow the claim. (Given
  the hard-decoy and paired-paraphrase results, this would be surprising.)
- `readable_overlap` = Jaccard of the tuned- and logit-readable failure sets; moderate-to-high overlap
  says the two lenses agree on which failures are readable.

In [ ]:
for nm,v in TUNED.items():
    if "status" in v: continue
    th=v["tuned_hard_readable_rate"]; nt=v["tuned_nottop_among_readable"]
    if th>0.05 and nt>0.8:
        verdict="dissociation PERSISTS under tuned lens (readable yet not selected)"
    elif th>0.0:
        verdict="smaller readable subset, dissociation persists within it"
    else:
        verdict="tuned lens removes readability -- narrow the claim"
    print(f"{nm.split('/')[-1]}: tuned_hard_readable={th:.3f}, nottop|readable={nt:.3f} -> {verdict}")

## Notes
- The tuned lens is fit on a HELD-OUT split (`N_FIT` items) disjoint from the eval items (`N_EVAL`), so
  the tuned read is not circular.
- This uses the translator form (A_l h_l + b_l -> h_final, then true final norm + unembedding), the
  standard tuned-lens formulation. `recon_R2` reports translator quality; if it is low for a layer band,
  raise `N_FIT` or `L2`, or restrict the band.
- Everything except the read is identical to the logit-lens pipeline: same failed items, gold targets,
  decoy pools, hard-decoy criterion, final rank.
- Two models only (Llama-3.1-8B, Qwen2.5-3B) by design -- this is a representative faithfulness check,
  not a full sweep.